# Constraint-reconstruction benchmark

Drives [`benchmarks/run.py`](run.py): can an engine recover a joint distribution
from **many noisy low-order statistics**, never seeing rows? Each *cell* fits one
engine on one noisy constraint bag and scores it against a held-out split
(see [README](README.md)).

- **`heldout_nll`** (headline) — nats/row on test rows; lower is better. The
  `independent` engine (product of noisy marginals) is the null an engine must beat.
- **`pair_tv_unseen`** — TV error on pairwise marginals *no constraint touched*:
  does the inductive bias fill in unconstrained correlations sensibly?

> **Cost note.** `independent` fits are instant. The **tensor-chain** (`tn`) fit
> is dominated by a one-time JAX/XLA **compile** (~15-30 s); each optimizer step
> is only a few ms. `sweep_tn_fast` (below) pays that compile **once** for the
> whole `tn` sub-grid — it holds the constraint *structure* fixed and refits only
> the noisy target values through a reusable batched loss — so extra `tn` cells
> cost ~seconds each, not another compile. Bump `TN_NOISE` / `TN_SEEDS` freely.

In [ ]:
import os, sys, time
os.environ.setdefault("JAX_PLATFORMS", "cpu")   # avoid GPU init noise
from pathlib import Path

# Make the `benchmarks` package importable and results paths resolve, wherever
# the kernel's cwd happens to be.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "benchmarks" / "run.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Persistent XLA compile cache: re-running an identical cell (e.g. after a kernel
# restart) loads the compiled executable from disk instead of recompiling.
import jax
jax.config.update("jax_compilation_cache_dir", str(ROOT / ".jax_cache"))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)

from benchmarks.datasets import DATASETS
from benchmarks.encoding import TableEncoder
from benchmarks.engines import IndependentEngine, TensorChainEngine, TensorTreeEngine
from benchmarks.run import run_cell, sweep_tn_fast, RESULTS_DIR

pd.set_option("display.width", 120)
print("repo root :", ROOT)
print("datasets  :", list(DATASETS))
print("jax backend:", jax.default_backend())

## Configuration

`QUICK=True` keeps the whole notebook to a few minutes. Set `QUICK=False` (and
edit the grids) for a paper-grade sweep.

In [ ]:
QUICK = True

DATASET   = "synthetic_chain"   # or "adult" (downloads via OpenML on first run)
N_BINS    = 8                   # bins per continuous column
MAX_ROWS  = 4000 if QUICK else None
SAVE_JSONL = False              # also append rows to results/<dataset>.jsonl (like run.py)

# Sweep grid. `independent` is instant, so it covers the full grid.
N_PAIR   = 20
CONFLICT = 0.0
NOISE    = [0.0, 0.15, 0.3, 0.6, 1.0]      # LLM logit-sd lives ~0.3-1.0
SEEDS    = [0, 1, 2]

# Tensor chain: one compile serves the whole sub-grid (sweep_tn_fast), so this
# can be generous. `TN_SEEDS` reseeds the noise + model init only — the constraint
# *structure* is held fixed across the sweep (that's what enables the single compile).
RUN_TN   = True
TN_NOISE = [0.0, 0.3, 1.0] if QUICK else NOISE
TN_SEEDS = [0, 1]          if QUICK else SEEDS
TN_STEPS = 400             # optimizer steps (~few ms each)
TN_BOND  = 6

# Tensor tree: variables at the leaves of a balanced binary latent tree. There is
# no compile-once fast sweep for the tree (sweep_tn_fast walks a linear chain), so
# each (noise, seed) cell pays its own ~20-30 s compile via run_cell — keep this
# sub-grid small under QUICK. kind="nonneg" routes scoring through the batched
# SteinerMarginals/BPPlan sweep; set TREE_KIND="born" for a same-kind comparison
# against the chain above.
RUN_TREE   = True
TREE_KIND  = "nonneg"
TREE_NOISE = [0.0, 0.3, 1.0] if QUICK else NOISE
TREE_SEEDS = [0]             if QUICK else SEEDS
TREE_STEPS = 400
TREE_BOND  = 6

print(f"independent: {len(NOISE) * len(SEEDS)} cells (instant)")
print(f"tensor-chain: {len(TN_NOISE) * len(TN_SEEDS) if RUN_TN else 0} cells "
      f"(one ~15-30s compile, then ~seconds each)")
print(f"tensor-tree: {len(TREE_NOISE) * len(TREE_SEEDS) if RUN_TREE else 0} cells "
      f"(~20-30s compile EACH — no fast sweep)")

## Encode the table

One shared discrete grid so every engine's scores are comparable.

In [ ]:
kw = {"max_rows": MAX_ROWS} if MAX_ROWS else {}
train, test = DATASETS[DATASET](**kw)
enc = TableEncoder(train, n_bins_cont=N_BINS)
Xi_train, Xi_test = enc.bin_indices(train), enc.bin_indices(test)
print(f"{DATASET}: {len(train)} train / {len(test)} test rows, "
      f"{len(enc.names)} vars, dims={enc.dims}")

## Run the sweep

`independent` runs cell-by-cell (instant). The tensor chain goes through
`sweep_tn_fast`, which compiles **once** and then refits each `(noise, seed)`
cell in seconds — watch the first `tn` line pay the compile and the rest fly.
The tensor **tree** (`RUN_TREE`) has no such fast path — each cell recompiles —
so its sub-grid (`TREE_NOISE` × `TREE_SEEDS`) is kept small under `QUICK`.

In [ ]:
rows = []
t_all = time.perf_counter()

# independent null — instant, full grid
print("independent:")
ind = IndependentEngine()
for noise in NOISE:
    for seed in SEEDS:
        row = run_cell(ind, enc, Xi_train, Xi_test,
                       n_pair=N_PAIR, noise=noise, conflict=CONFLICT, seed=seed)
        rows.append(row)

# tensor chain — one compile for the whole sub-grid
if RUN_TN:
    print("tensor-chain (first cell compiles, rest reuse):")
    tn = TensorChainEngine(bond_dim=TN_BOND, steps=TN_STEPS)
    rows += sweep_tn_fast(tn, enc, Xi_train, Xi_test, n_pair=N_PAIR,
                          noise_grid=TN_NOISE, seeds=TN_SEEDS, conflict=CONFLICT)

# tensor tree — no fast sweep, so run_cell per (noise, seed): each cell compiles
if RUN_TREE:
    print("tensor-tree (each cell compiles ~20-30s):")
    tree = TensorTreeEngine(bond_dim=TREE_BOND, kind=TREE_KIND, steps=TREE_STEPS)
    for noise in TREE_NOISE:
        for seed in TREE_SEEDS:
            row = run_cell(tree, enc, Xi_train, Xi_test, n_pair=N_PAIR,
                           noise=noise, conflict=CONFLICT, seed=seed)
            rows.append(row)
            print(f"  {row['engine']:>14}  noise={noise:<4} seed={seed}  "
                  f"nll={row['heldout_nll']:.4f}  "
                  f"pair_tv_unseen={row['pair_tv_unseen']:.4f}  "
                  f"({row['fit_seconds']}s)")

df = pd.DataFrame(rows)

if SAVE_JSONL:
    RESULTS_DIR.mkdir(exist_ok=True)
    out = RESULTS_DIR / f"{DATASET}.jsonl"
    with open(out, "a") as fh:
        for r in rows:
            fh.write(__import__("json").dumps(r) + "\n")
    print(f"appended {len(rows)} rows to {out}")

print(f"\ndone in {time.perf_counter() - t_all:.1f}s")

## Summary table

Mean over seeds for each `(engine, noise)` cell.

In [ ]:
metrics = ["heldout_nll", "pair_tv_unseen", "pair_tv_seen", "marginal_tv"]
summary = (df.groupby(["engine", "noise"])[metrics]
             .mean().round(4).reset_index())
summary

## Curves that matter

`heldout_nll` vs noise (does the engine stay ahead of the `independent` null as
constraint estimates degrade?) and `pair_tv_unseen` vs noise (does it generalise
to correlations no constraint pinned down?).

In [ ]:
agg = df.groupby(["engine", "noise"]).agg(
    nll_mean=("heldout_nll", "mean"), nll_std=("heldout_nll", "std"),
    tv_mean=("pair_tv_unseen", "mean"), tv_std=("pair_tv_unseen", "std"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for (metric, err, title, ylab), ax in zip(
    [("nll_mean", "nll_std", "Held-out NLL vs noise", "heldout_nll (nats/row)"),
     ("tv_mean", "tv_std", "Unseen-pair TV vs noise", "pair_tv_unseen")], axes):
    for name, g in agg.groupby("engine"):
        g = g.sort_values("noise")
        ax.errorbar(g["noise"], g[metric], yerr=g[err].fillna(0.0),
                    marker="o", capsize=3, label=name)
    ax.set(title=title, xlabel="prob_logit_sd (noise)", ylabel=ylab)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"{DATASET}  |  n_pair={N_PAIR}, conflict={CONFLICT}", y=1.02)
fig.tight_layout()
plt.show()

## Where to go next

- **Fuller `tn` curve:** set `QUICK = False` (or `TN_NOISE = NOISE`, `TN_SEEDS = SEEDS`).
  Extra cells reuse the one compile, so they cost ~seconds each.
- **`nll` vs `n_pair`** (value of each constraint) or **vs `conflict`** (robustness):
  make `N_PAIR` / `CONFLICT` the swept axis. Note `sweep_tn_fast` fixes the
  constraint *structure*, so vary `N_PAIR` across separate `sweep_tn_fast` calls
  (each is its own compile) rather than inside one.
- **More engines:** implement `fit()` on `GaussianEngine` / `PCEngine` in
  [`engines.py`](engines.py) and add an `independent`-style loop for them.
- **Load a prior sweep** instead of refitting:
  ```python
  df = pd.read_json(RESULTS_DIR / f"{DATASET}.jsonl", lines=True)
  ```